<a href="https://colab.research.google.com/github/iDurugkar/practice-2026/blob/main/TorchCode/10_gqa.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/10_gqa.ipynb)

# 🔴 Hard: Grouped Query Attention (GQA)

Implement **Grouped Query Attention** — used in LLaMA 2, Mistral, etc. to reduce KV cache size.

Like MHA, but with **fewer KV heads** than Q heads. Each group of Q heads shares the same K/V head.

### Signature
```python
class GroupQueryAttention:
    def __init__(self, d_model: int, num_heads: int, num_kv_heads: int): ...
    def forward(self, x) -> torch.Tensor:  # self-attention
```

### Requirements
- `self.W_q`: `nn.Linear(d_model, d_model)` — full Q projection
- `self.W_k`: `nn.Linear(d_model, num_kv_heads * d_k)` — reduced K projection
- `self.W_v`: `nn.Linear(d_model, num_kv_heads * d_k)` — reduced V projection
- `self.W_o`: `nn.Linear(d_model, d_model)` — output projection
- `d_k = d_model // num_heads`
- Expand KV heads with `repeat_interleave` to match Q heads
- When `num_kv_heads == num_heads`, should behave like standard MHA

In [1]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.5/48.5 kB 2.0 MB/s eta 0:00:00


In [3]:
import torch
import torch.nn as nn
import math

In [12]:
# ✏️ YOUR IMPLEMENTATION HERE

class GroupQueryAttention:
    def __init__(self, d_model, num_heads, num_kv_heads):
        assert d_model % num_heads == 0
        self.num_heads = num_heads
        self.d_model = d_model
        self.d_k = d_model // num_heads
        self.num_kv_heads = num_kv_heads

        self.W_k = nn.Linear(d_model, self.d_k * self.num_kv_heads)
        self.W_v = nn.Linear(d_model, self.d_k * self.num_kv_heads)
        self.W_q = nn.Linear(d_model, d_model)
        self.W_o = nn.Linear(d_model, d_model)


    def forward(self, x):
        B, S, d_model = x.shape
        assert d_model == self.d_model
        q = self.W_q(x).view(B, S, self.num_heads, self.d_k).transpose(1, 2)

        v = self.W_v(x).view(B, S, self.num_kv_heads, self.d_k).repeat_interleave(self.num_heads // self.num_kv_heads, axis=2)
        k = self.W_k(x).view(B, S, self.num_kv_heads, self.d_k).repeat_interleave(self.num_heads // self.num_kv_heads, axis=2)

        v = v.transpose(1, 2)
        k = k.transpose(1, 2)
        logits = torch.matmul(q, k.transpose(-1, -2)) / torch.sqrt(torch.tensor(self.d_k, dtype=torch.float32))
        attn = torch.softmax(logits, axis=-1)
        result = torch.matmul(attn, v)
        result = result.transpose(1, 2).contiguous().view(B, S, self.d_model)
        return result

In [13]:
# 🧪 Debug
torch.manual_seed(0)
gqa = GroupQueryAttention(d_model=32, num_heads=8, num_kv_heads=2)
print("W_q shape:", gqa.W_q.weight.shape)  # (32, 32)
print("W_k shape:", gqa.W_k.weight.shape)  # (8, 32)  — only 2 KV heads * d_k=4

x = torch.randn(2, 6, 32)
out = gqa.forward(x)
print("Output shape:", out.shape)           # (2, 6, 32)

W_q shape: torch.Size([32, 32])
W_k shape: torch.Size([8, 32])
Output shape: torch.Size([2, 6, 32])


In [14]:
from torch_judge import check
check('gqa')


🧪 Testing: Grouped Query Attention (Hard)
──────────────────────────────────────────────────
  ✅ [1/5] Output shape (2.3ms)
  ✅ [2/5] nn.Linear with correct shapes (0.5ms)
  ✅ [3/5] Degenerates to MHA when kv_heads == heads (1.2ms)
  ✅ [4/5] KV heads are shared correctly (5.8ms)
  ✅ [5/5] Gradient flow (32.6ms)
──────────────────────────────────────────────────
  🎉 All 5 tests passed! (42.4ms total)
  Progress saved. Run status() to see your dashboard.

